# Day 27 — Geospatial basics (GeoPandas) or domain-specific viz
Objectives:
- If GeoPandas available, plot simple geospatial data.
- Otherwise, pick a domain visualization (e.g., networkx).
Note: GeoPandas may require system dependencies; code below is illustrative.

<!-- BEGIN BEGINNER NOTEBOOK DEEP DIVE -->
## How to use this notebook

This is the editable learner artifact for `python-27`. Read
`python/ds-60day/companion-guides/day27_geospatial_or_domain_viz.md` first, then work here with the **Python (ds60sqlpy)**
kernel. Restart the kernel and run from top to bottom so an earlier
hidden value cannot make later code appear correct.

For every example: (1) write a prediction, (2) run the cell,
(3) compare the exact value, type, shape, rows, or side effect with
the stated observation, and (4) explain one mismatch before moving
on. For every exercise, use its dedicated work cell and include a
real assertion or bounded inspection. The official solution stays
closed until you have a tested attempt.

The notebook is deliberately offline after course setup. Do not add
`%pip`, credentials, absolute developer paths, or shell-specific
setup here. If an import fails, use the repository doctor and the
catalog from a terminal rather than changing only this kernel.

## Core mental model

Geospatial data combines ordinary attributes with geometry and a
coordinate reference system (CRS). A coordinate pair is conventionally
`(x, y)`—longitude then latitude for geographic coordinates. Values can
look numerically valid while being semantically swapped, so record
source order and bounds.

A CRS defines what coordinates mean and their units. Layers cannot be
overlaid or spatially joined honestly until their CRSs are known and
compatible. Reprojection transforms coordinates; assigning a CRS merely
labels existing coordinates and is not a substitute. Inspect validity,
emptiness, bounds, provenance, and offline availability before plotting.

### Vocabulary

- **geometry:** a spatial object such as point, line, or polygon.
- **CRS:** coordinate reference system defining coordinate meaning and units.
- **reprojection:** transforming geometry coordinates from one CRS to another.
- **longitude:** east-west angular coordinate, used as x in geographic pairs.
- **latitude:** north-south angular coordinate, used as y in geographic pairs.
- **spatial join:** matching records by a geometric relationship such as within or intersects.

## Syntax anatomy

`GeoDataFrame(data, geometry=..., crs="EPSG:4326")` declares that the
supplied longitude/latitude coordinates already use WGS 84.
`.to_crs("EPSG:3857")` transforms them to Web Mercator meters. In
contrast, `.set_crs(...)` labels coordinates without moving them and
should be used only when the original CRS is known but missing.

### Worked example 1 — Create known geographic points and inspect bounds

Constructed geometry avoids any network or external file. Before running the next cell, predict its final displayed
value and identify the line responsible for every intermediate.

In [ ]:
import geopandas as gpd
from shapely.geometry import Point

places = gpd.GeoDataFrame(
    {"name": ["A", "B"]},
    geometry=[Point(-122.4, 37.8), Point(-73.9, 40.7)],
    crs="EPSG:4326",
)
(str(places.crs), places.total_bounds.round(1).tolist())

**Expected observation:** The CRS is WGS 84/EPSG:4326 and bounds are approximately `[-122.4, 37.8, -73.9, 40.7]` in longitude/latitude degrees.

If your result differs, compare inputs and types before rerunning.
Then explain the example from input to evidence in your own words.

### Worked example 2 — Reproject rather than relabel

Projected coordinates use different units while representing the same places. Predict first; then run the next cell.

In [ ]:
projected = places.to_crs("EPSG:3857")
{
    "same_rows": len(projected) == len(places),
    "projected_crs": str(projected.crs),
    "x_units_are_not_degrees": abs(projected.geometry.x.iloc[0]) > 1_000,
}

**Expected observation:** All checks are true; the coordinates are transformed to meter-like Web Mercator values.

## Debugging clinic

When evidence differs from your prediction, use this order:

1. Inspect `.crs`, geometry types, `.is_valid`, `.is_empty`, and `.total_bounds` before plotting.
2. Verify coordinate order with known places or realistic bounds.
3. Use `.to_crs` to transform; use `.set_crs` only to label coordinates whose source CRS is known.
4. Keep an offline analytical layer or domain-graph alternative when basemap data is unavailable.

**Alternative to compare:** When location is not central or source geometry is unavailable, a deterministic domain/network visualization can teach graph structure without a misleading map.

**Boundary to test:** Missing CRS, invalid/self-intersecting geometry, dateline crossing, swapped axes, empty geometry, and remote basemap failure need policy.

Do not move on merely because the cell runs. Explain which object,
branch, axis, row, or resource changed and why.

In [ ]:
# This optional example downloads and caches Natural Earth data on first use.
try:
    import geopandas as gpd
    from geodatasets import get_path

    world = gpd.read_file(get_path('naturalearth.land'))
    world.plot(figsize=(8, 6), edgecolor='black')
except ImportError:
    print('Install the geo dependency group to run this optional example.')


## Exercises and progressive hints

Each item is a complete mini-contract. Before writing code, copy its input,
expected behavior, constraints, and verification into your work cell. A
result is not complete merely because it “looks right”; run the stated
assertion or inspection and explain what it proves.

1. Using only a local or already-cached boundary/point source, create a GeoDataFrame plot. **Before plotting:** record provenance/license, CRS, geometry types, invalid/empty counts, and bounds. **Constraints:** transform all layers to one appropriate CRS and never use `set_crs` as if it reprojected coordinates.
   **Verify:** check a known location/bounds and confirm entity count survives reprojection.

2. Alternatively, construct a deterministic NetworkX domain graph where node and edge meanings are written explicitly. **Constraints:** use a fixed layout seed, encode at most a few meaningful attributes, provide labels/legend, and do not require network data.
   **Verify:** reconcile plotted node/edge counts to the graph and explain what spatial questions this fallback cannot answer.

### Additional mastery practice

Treat coordinate reference system, coordinate order, source provenance, geometry validity, and network/cache behavior as part of the visualization.

Continue with five new exercises. Record each prediction before running
code; these extend rather than replace the original practice above.

3. **Prediction:** Predict what goes wrong when latitude/longitude degrees are overlaid on a web map measured in Web Mercator meters.
   **Progressive hint:** Layers must use compatible coordinate reference systems (CRSs).
   **Verify:** Compare both CRS/bounds, then assert reprojection makes units compatible and the transformed layers overlap in a plausible extent.
4. **Tracing:** Trace one point represented as `(longitude, latitude)` and explain why swapping values can still create a valid but wrong location.
   **Progressive hint:** Both numbers may fall in legal ranges, so semantic order matters.
   **Verify:** Validate the known longitude/latitude pair against real bounds, swap it, and show why semantic checks—not only numeric ranges—detect the wrong location.
5. **Implementation:** Implement a longitude/latitude bounding-box validator with clear ordering and range checks.
   **Progressive hint:** Require west ≤ east, south ≤ north, and geographic bounds.
   **Verify:** Assert a valid box passes and separately reject west>east, south>north, longitude outside ±180, and latitude outside ±90.
6. **Debugging:** Repair a spatial join or overlay attempted before CRS comparison and reprojection.
   **Progressive hint:** Inspect `.crs`; transform one layer to the other's CRS.
   **Verify:** Show the pre-reprojection CRS mismatch, transform one layer, and assert the operation now uses equal CRS while preserving row identities.
7. **Edge case and explanation:** Define fallback behavior for missing/invalid geometry or unavailable map data, including an offline domain-graph alternative.
   **Progressive hint:** A missing basemap should not erase the analytical data layer.
   **Verify:** Feed missing/invalid geometry and unavailable basemap fixtures; assert analytical records remain represented by the documented local layer or graph fallback.

Before opening the reference solution, write one sentence explaining
which contract or mental model each result confirms.

### Practice 1 — prediction, attempt, and evidence

**Contract reminder:** Using only a local or already-cached boundary/point source, create a GeoDataFrame plot. **Before plotting:** record provenance/license, CRS, geometry types, invalid/empty counts, and bounds. **Constraints:** transform all layers to one appropriate CRS and never use `set_crs` as if it reprojected coordinates. **Verify:** check a known location/bounds and confirm entity count survives reprojection.

In the next cell, record your prediction before the code. Keep
the input tiny, implement only this contract, and finish with
the requested assertion or bounded inspection. If it fails,
retain the smallest failing input and write what the evidence
changed about your hypothesis.

In [ ]:
# Practice 1 — your work
# Short contract: Using only a local or already-cached boundary/point source, create a GeoDataFrame plot. record provenance/license, CRS, geometry types, invalid/empty counts, and bounds. transfo...
# Prediction:

# Implementation:

# Verification (assert or inspect exactly what the prompt requires):


### Practice 2 — prediction, attempt, and evidence

**Contract reminder:** Alternatively, construct a deterministic NetworkX domain graph where node and edge meanings are written explicitly. **Constraints:** use a fixed layout seed, encode at most a few meaningful attributes, provide labels/legend, and do not require network data. **Verify:** reconcile plotted node/edge counts to the graph and explain what spatial questions this fallback cannot answer.

In the next cell, record your prediction before the code. Keep
the input tiny, implement only this contract, and finish with
the requested assertion or bounded inspection. If it fails,
retain the smallest failing input and write what the evidence
changed about your hypothesis.

In [ ]:
# Practice 2 — your work
# Short contract: Alternatively, construct a deterministic NetworkX domain graph where node and edge meanings are written explicitly. use a fixed layout seed, encode at most a few meaningful attr...
# Prediction:

# Implementation:

# Verification (assert or inspect exactly what the prompt requires):


### Practice 3 — prediction, attempt, and evidence

**Contract reminder:** **Prediction:** Predict what goes wrong when latitude/longitude degrees are overlaid on a web map measured in Web Mercator meters. **Progressive hint:** Layers must use compatible coordinate reference systems (CRSs). **Verify:** Compare both CRS/bounds, then assert reprojection makes units compatible and the transformed layers overlap in a plausible extent.

In the next cell, record your prediction before the code. Keep
the input tiny, implement only this contract, and finish with
the requested assertion or bounded inspection. If it fails,
retain the smallest failing input and write what the evidence
changed about your hypothesis.

In [ ]:
# Practice 3 — your work
# Short contract: Predict what goes wrong when latitude/longitude degrees are overlaid on a web map measured in Web Mercator meters. Layers must use compatible coordinate reference systems (CRSs)...
# Prediction:

# Implementation:

# Verification (assert or inspect exactly what the prompt requires):


### Practice 4 — prediction, attempt, and evidence

**Contract reminder:** **Tracing:** Trace one point represented as `(longitude, latitude)` and explain why swapping values can still create a valid but wrong location. **Progressive hint:** Both numbers may fall in legal ranges, so semantic order matters. **Verify:** Validate the known longitude/latitude pair against real bounds, swap it, and show why semantic checks—not only numeric ranges—detect the wrong location.

In the next cell, record your prediction before the code. Keep
the input tiny, implement only this contract, and finish with
the requested assertion or bounded inspection. If it fails,
retain the smallest failing input and write what the evidence
changed about your hypothesis.

In [ ]:
# Practice 4 — your work
# Short contract: Trace one point represented as `(longitude, latitude)` and explain why swapping values can still create a valid but wrong location. Both numbers may fall in legal ranges, so sem...
# Prediction:

# Implementation:

# Verification (assert or inspect exactly what the prompt requires):


### Practice 5 — prediction, attempt, and evidence

**Contract reminder:** **Implementation:** Implement a longitude/latitude bounding-box validator with clear ordering and range checks. **Progressive hint:** Require west ≤ east, south ≤ north, and geographic bounds. **Verify:** Assert a valid box passes and separately reject west>east, south>north, longitude outside ±180, and latitude outside ±90.

In the next cell, record your prediction before the code. Keep
the input tiny, implement only this contract, and finish with
the requested assertion or bounded inspection. If it fails,
retain the smallest failing input and write what the evidence
changed about your hypothesis.

In [ ]:
# Practice 5 — your work
# Short contract: Implement a longitude/latitude bounding-box validator with clear ordering and range checks. Require west ≤ east, south ≤ north, and geographic bounds. Assert a valid box passes...
# Prediction:

# Implementation:

# Verification (assert or inspect exactly what the prompt requires):


### Practice 6 — prediction, attempt, and evidence

**Contract reminder:** **Debugging:** Repair a spatial join or overlay attempted before CRS comparison and reprojection. **Progressive hint:** Inspect `.crs`; transform one layer to the other's CRS. **Verify:** Show the pre-reprojection CRS mismatch, transform one layer, and assert the operation now uses equal CRS while preserving row identities.

In the next cell, record your prediction before the code. Keep
the input tiny, implement only this contract, and finish with
the requested assertion or bounded inspection. If it fails,
retain the smallest failing input and write what the evidence
changed about your hypothesis.

In [ ]:
# Practice 6 — your work
# Short contract: Repair a spatial join or overlay attempted before CRS comparison and reprojection. Inspect `.crs`; transform one layer to the other's CRS. Show the pre-reprojection CRS mismatch...
# Prediction:

# Implementation:

# Verification (assert or inspect exactly what the prompt requires):


### Practice 7 — prediction, attempt, and evidence

**Contract reminder:** **Edge case and explanation:** Define fallback behavior for missing/invalid geometry or unavailable map data, including an offline domain-graph alternative. **Progressive hint:** A missing basemap should not erase the analytical data layer. **Verify:** Feed missing/invalid geometry and unavailable basemap fixtures; assert analytical records remain represented by the documented local layer or graph fallback.

In the next cell, record your prediction before the code. Keep
the input tiny, implement only this contract, and finish with
the requested assertion or bounded inspection. If it fails,
retain the smallest failing input and write what the evidence
changed about your hypothesis.

In [ ]:
# Practice 7 — your work
# Short contract: Define fallback behavior for missing/invalid geometry or unavailable map data, including an offline domain-graph alternative. A missing basemap should not erase the analytical d...
# Prediction:

# Implementation:

# Verification (assert or inspect exactly what the prompt requires):
